# [Chapter 13 — Sexual Transmission, §13.2] The Bipartite Heterosexual $SIS$ Model

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 13 — Sexual Transmission
**Considerations developed:** 4 (force of infection direction), 7 (mixing balance)
**Estimated runtime:** ≤ 20 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_13_sexual_transmission/01_bipartite_STI_model.ipynb)


## What this notebook does

Implements the canonical bipartite (M↔F) $SIS$ model used for curable sexually transmitted infections. Sexual transmission is structurally different from respiratory transmission: contacts are *between* groups by definition, not within (Consideration~4 — direction matters), and the partner-change rates $c_{MF}$ and $c_{FM}$ are constrained by partnership symmetry (Consideration~7).

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_13
from shared.verification import assert_within_tolerance
set_seed_chapter_13()
book_style()


## Bipartite $SIS$ system

$$\dot S_M = -\beta_{MF} c_{MF} S_M I_F/N_F + \gamma I_M, \qquad \dot I_M = +(\ldots) - \gamma I_M$$
and similarly for the female compartment with $M \leftrightarrow F$. The partnership-symmetry constraint is $c_{MF} N_M = c_{FM} N_F$.

In [ ]:
from shared.parameters import baseline_chapter_13
p = baseline_chapter_13()

# Verify partnership symmetry
lhs = p['cMF'] * p['NM']
rhs = p['cFM'] * p['NF']
print(f'Symmetry: c_MF N_M = {lhs:.3f}, c_FM N_F = {rhs:.3f}')
assert_within_tolerance(lhs, rhs, abs_tol=1e-9, label='partnership symmetry')

def sis_rhs(y, t, p):
    SM, IM, SF, IF = y
    NM, NF = p['NM'], p['NF']
    gamma = 1.0/p['tau_R']
    inc_M = p['betaMF'] * p['cMF'] * SM * IF / NF
    inc_F = p['betaFM'] * p['cFM'] * SF * IM / NM
    return [-inc_M + gamma*IM, inc_M - gamma*IM,
            -inc_F + gamma*IF, inc_F - gamma*IF]

y0 = [p['SM0'], p['IM0'], p['SF0'], p['IF0']]
t = np.linspace(p['t_start'], p['t_end'], p['n_points'])
sol = odeint(sis_rhs, y0, t, args=(p,))
SM, IM, SF, IF = sol.T


## Geometric-mean reproductive number

For the bipartite $SIS$, the next-generation matrix has zeros on the diagonal (M does not infect M directly), so the dominant eigenvalue is the geometric mean of the cross-transmission products:
$$\mathcal{R}_0 = \sqrt{\,c_{MF}\,c_{FM}\,\beta_{MF}\,\beta_{FM}\,\tau_R^2\,}.$$

In [ ]:
import math
R0 = math.sqrt(p['cMF']*p['cFM']*p['betaMF']*p['betaFM']*p['tau_R']**2)
print(f'R_0 (geometric mean) = {R0:.4f}')
print(f'  -> below threshold; expect both prevalences to decay to zero.')


## Trajectory under sub-threshold parameters

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t, IM/p['NM'], color=BOOK_COLORS['primary'], lw=2, label='Male prevalence')
ax.plot(t, IF/p['NF'], color=BOOK_COLORS['secondary'], lw=2, label='Female prevalence')
ax.set_xlabel('time (years)')
ax.set_ylabel('prevalence within group')
ax.set_title(f'Bipartite SIS, R_0 = {R0:.2f} (sub-threshold)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

assert IM[-1] + IF[-1] < IM[0] + IF[0], 'sub-threshold: prevalence must decline'
print('Verified: sub-threshold parameters produce decay.')


## Push above threshold

Increasing $c_{MF}=c_{FM}$ to 4 (with proportional rebalancing) lifts $\mathcal{R}_0$ above 1 and produces a non-trivial endemic equilibrium.

In [ ]:
p2 = dict(p)
p2['cMF'] = p2['cFM'] = 4.0
p2['t_end'] = 100.0
p2['n_points'] = 2001
R0_2 = math.sqrt(p2['cMF']*p2['cFM']*p2['betaMF']*p2['betaFM']*p2['tau_R']**2)
print(f'New R_0 = {R0_2:.3f}')

y0_2 = [p2['SM0'], p2['IM0'], p2['SF0'], p2['IF0']]
t2 = np.linspace(p2['t_start'], p2['t_end'], p2['n_points'])
sol2 = odeint(sis_rhs, y0_2, t2, args=(p2,))
SM2, IM2, SF2, IF2 = sol2.T

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t2, IM2/p2['NM'], color=BOOK_COLORS['primary'], lw=2, label='Male')
ax.plot(t2, IF2/p2['NF'], color=BOOK_COLORS['secondary'], lw=2, label='Female')
ax.set_xlabel('time (years)')
ax.set_ylabel('prevalence')
ax.set_title(f'Bipartite SIS, R_0 = {R0_2:.2f} (super-threshold endemic)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## What's next

- Notebook 02 explores how *assortative* mixing (preferential within-activity-class partnerships) changes $\mathcal{R}_0$ for STIs.
- Chapter 19 builds on this with HIV-specific extensions (latency, treatment as prevention).
- The geometric-mean form generalizes to any bipartite cycle of length 2; vector-host models in Chapter 14 use the same algebra.